## Golden Model for NMS Algorithm

Uses File I/O Golden Model for hardware verification inspired from this [article](https://thedatabus.in/python-iverilog-verification/)

**This notebook is narrative, not the implementation.** The float `calculate_iou`/`nms` below are
kept exactly as originally written, purely as the "as originally written" comparison baseline —
see `models/bench_cpu.py`, which times it against the real thing. From the "Integer golden model"
section onward, everything imports `models/nms_model.py`, which is the actual golden model the
RTL is checked against (see `docs/NMS.md` and `docs/architecture.md`'s frozen interface spec).

The input were considering has these parameters
- The confidence score `c`
- Lower left cordinate `(x, y)`
- Upper right coordinate `(a, b)`

In [ ]:
x1, y1, a1, b1, c1 = 200, 300, 400, 500, 0.86  # point 1
x2, y2, a2, b2, c2 = 350, 350, 450, 600, 0.65  # point 2

The 12 initial representation of coodinates comes from UART bit alignment. Take 12 bit for granted and do the calculations

Bounding box rejection criteria
$$\text{Area\_Overlap} \times 2^k \geq \text{Threshold\_INT} \times Area\_Union$$

| variable | Signed / Unsigned | Integer Bits | Fractional Bits | Total Width | Domain max (1080p) | Representable Range |
| --- | --- | --- | --- | --- | --- | --- |
| x/y/a/b | u | 12 | 0 | 12 | 1920 | 0 to 4095 |
| area1/area2 | u | 24 | 0 | 24 | 2073600 | 0 to 16777215 |
| xx/yy/aa/bb | u | 12 | 0 | 12 | 1920 | 0 to 4095 |
| t_w/t_h | s | 13 | 0 | 13 | 1920 | -4096 to +4095
| w/h | u | 12 | 0 | 12 | 1920 | 0 to 4095 | # can clamp it back
| intersection_area | u | 24 | 0 | 24 | 2073600 | 0 to 16777215 |
|  t_union_area | s | 26 | 0 | 26 | 2073600 | -33554432 to 33554431 | # positive side dominates thus to account for sign, we need 26 bits (assumes that the 3 data paths are independent)
|  union_area | u | 25 | 0 | 25 | 2073600 | 0 to 33554431 | # can clamp it back to 25 bits
|  T_INT | u | 8 | 0 | 8 | 255 | 0 to 255 | # Q0.8 fixed point; the project's own threshold is T_INT=128 (0.5), which needs the full 0-255 range, not 0-126

`2^k` (`k=8`) is not a stored field — it's a shift amount compiled into the RTL as `I << 8`, so it
has no row of its own here. See `docs/plan.md` Q11 for the full correction; this table matches
`models/nms_params.py`, the single source of truth these widths are pulled from.</cell id="3">


In [ ]:
# Calculations needed for IoU

area1 = (a1 - x1) * (b1 - y1)
area2 = (a2 - x2) * (b2 - y2)

# maximize the lower left and minimize the upper right (nature of intersection)
xx = max(x1, x2)
yy = max(y1, y2)
aa = min(a1, a2)
bb = min(b1, b2)

# width and height
t_w = aa - xx
t_h = bb - yy
w = max(0, t_w)
h = max(0, t_h)

# intersection and union area
intersection_area = w * h
union_area = area1 + area2 - intersection_area
iou = intersection_area / union_area

In [ ]:
iou

In [ ]:
bbox1 = 200, 300, 400, 500, 0.85
bbox2 = 350, 350, 450, 600, 0.65

### As originally written — float baseline

`calculate_iou` and `nms` below are unchanged from the original notebook: float division,
`iou_threshold` as a plain float. They are **not** the golden model — kept only as the "as
written" comparison point that `models/bench_cpu.py` benchmarks against the integer model and a
vectorised numpy version (`docs/plan.md` Part 1e).

In [ ]:
def calculate_iou(bbox1: tuple, bbox2: tuple) -> float:
    """Calculate IoU value from bounding boxes.

    Assumes each bounding box is a `tuple` follows this structure
    `(lower left cordinate, upper right cordinate, confidence_score).

    Args:
        bbox1 (tuple): bounding box 1
        bbox2 (tuple): bounding box 2

    Returns:
        float: calulated IoU value
    """
    x1, y1, a1, b1, _ = bbox1
    x2, y2, a2, b2, _ = bbox2

    # Calculations needed for IoU
    area1 = (a1 - x1) * (b1 - y1)
    area2 = (a2 - x2) * (b2 - y2)

    # maximize the lower left and minimize the upper right (nature of intersection)
    xx = max(x1, x2)
    yy = max(y1, y2)
    aa = min(a1, a2)
    bb = min(b1, b2)

    # width and height
    w = max(0, (aa - xx))
    h = max(0, (bb - yy))

    # intersection and union area
    intersection_area = w * h
    union_area = area1 + area2 - intersection_area
    iou = intersection_area / union_area

    return iou

In [ ]:
def nms(boxes: list, iou_threshold: float) -> list:
    """Run winner-takes-all NMS using float `calculate_iou`, as originally written.

    Args:
        boxes: `(x, y, a, b, confidence)` tuples.
        iou_threshold: Suppression threshold.

    Returns:
        The surviving boxes, most confident first.
    """
    sorted_boxes = sorted(boxes, key=lambda box: box[4], reverse=True)
    valid = [True] * len(sorted_boxes)
    keep = []

    for i in range(len(sorted_boxes)):
        if valid[i]:
            keep.append(sorted_boxes[i])
            valid[i] = False

            for j in range(i + 1, len(sorted_boxes)):
                if valid[j]:
                    iou = calculate_iou(sorted_boxes[i], sorted_boxes[j])
                    if iou >= iou_threshold:
                        valid[j] = False

    return keep

In [ ]:
# (x1, y1, x2, y2, confidence)
test_boxes = [
    # Cluster A — cat detection (~8 overlapping boxes)
    (10, 10, 50, 50, 0.95),
    (12, 11, 52, 51, 0.90),
    (9, 13, 48, 53, 0.85),
    (14, 9, 54, 49, 0.70),
    (11, 14, 51, 54, 0.65),
    (15, 12, 55, 52, 0.55),
    (8, 8, 46, 46, 0.40),
    (13, 15, 53, 55, 0.30),
    # Cluster B — dog detection (~8 overlapping boxes)
    (100, 100, 150, 150, 0.92),
    (102, 101, 152, 151, 0.88),
    (98, 103, 148, 153, 0.80),
    (104, 99, 154, 149, 0.72),
    (101, 105, 151, 155, 0.60),
    (103, 98, 153, 148, 0.50),
    (97, 102, 147, 152, 0.42),
    (105, 104, 155, 154, 0.28),
    # Cluster C — car detection (~8 overlapping boxes)
    (200, 50, 260, 100, 0.93),
    (202, 52, 262, 102, 0.87),
    (198, 48, 258, 98, 0.78),
    (204, 53, 264, 103, 0.68),
    (201, 47, 261, 97, 0.58),
    (199, 55, 259, 105, 0.48),
    (203, 49, 263, 99, 0.38),
    (197, 51, 257, 101, 0.25),
    # Cluster D — person detection (~5 overlapping boxes)
    (50, 200, 100, 280, 0.91),
    (52, 202, 102, 282, 0.82),
    (48, 198, 98, 278, 0.73),
    (54, 203, 104, 283, 0.62),
    (47, 199, 97, 279, 0.45),
    # Isolated boxes — should all survive
    (300, 300, 340, 340, 0.75),
    (0, 280, 30, 310, 0.35),
    (280, 0, 320, 40, 0.20),
]

In [ ]:
nms(test_boxes, 0.5)

## Integer golden model (the real thing)

Everything above is float and exists only as the comparison baseline. From here on, the box
coordinates are quantised to the same `Box(x, y, a, b, score)` the RTL will see, and the actual
suppression decision comes from `models/nms_model.py` — no floats, no division, the exact
`(I << 8) >= T_INT * U` predicate the hardware compares (`docs/architecture.md`'s frozen
interface spec). `models/gen_vectors.py`'s `case_notebook31()` builds this same dataset, and
`models/test_model.py` asserts the 7-survivor count below as its primary acceptance check.

In [ ]:
import sys

sys.path.insert(0, "models")

from nms_model import Box, nms_allpairs, nms_sequential, quantise_score

present_mask = (
    1 << len(test_boxes)
) - 1  # every slot in test_boxes is a real detection
integer_boxes = [Box(x, y, a, b, quantise_score(c)) for x, y, a, b, c in test_boxes]

keep_seq = nms_sequential(integer_boxes, present_mask)
keep_all = nms_allpairs(integer_boxes, present_mask)
assert keep_seq == keep_all, "sequential and all-pairs structures must agree (Q20)"

survivors = [box for i, box in enumerate(integer_boxes) if (keep_seq >> i) & 1]
summary = f"{len(survivors)} survivors (integer model, all-pairs == sequential)"
summary, survivors